# Grading behavior, not just the final answer

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 22 §22.3–§22.7 · type: walkthrough*

Grading only the shipped unit hides *where* the factory failed. Here you **decompose**:
RAG into retrieval-vs-generation, an agent run into its **trajectory** (tool selection,
argument values, efficiency, recovery, path-safety), and a multi-turn dialogue with a
**user-simulator** graded on its final side effects.

## 🧠 Why this matters

A pipeline that ships a wrong answer can fail in many places, and a single final-answer score
tells you only *that* it failed, never *where*. A RAG answer is wrong because retrieval missed
the chunk **or** because generation ignored a chunk it had — opposite fixes. An agent can
reach the right answer by accident, burn forty tool calls doing it, or pass through an unsafe
action on the way. **Decompose the verdict** so each score points at one fork: retriever vs
prompt, tool selection vs argument values, the path vs the destination. That is the difference
between an eval that *diagnoses* and one that merely *grades*.

## Objectives & prerequisites

**By the end you can:**
- split a **RAG** eval into retrieval (recall/precision@k, MRR) and generation (faithfulness, answer-relevance);
- score an agent **trajectory** — selection, **efficiency** vs a budget, **recovery**, **path-safety**;
- grade **tool-call accuracy** including the *argument values* (the book's `tool_call_score`);
- drive a multi-turn **user-simulator** (the `simulate` loop) and grade the **final side effects**;
- convert a **production signal** into a **PII-redacted** golden case.

**Prereqs:** `22-01` (scorers + judge); concepts from Ch 13 (RAG retrieval metrics) and Ch 16–17 (trajectories).

## Setup

Same contract as `22-01`: `MOCK=1` (default) makes the retriever, agent traces, judge, and
persona-simulator **all canned and seeded** — free, offline, deterministic. `MOCK=0` would
spend a handful of judge + simulator turns per case (use a *cheaper* model for the simulator).

In [ ]:
import json
import os
import random
import pathlib
from collections import defaultdict

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

MOCK = os.getenv("COMPANION_MOCK", "1") == "1"
random.seed(22)

DATA = pathlib.Path("data")

def load_jsonl(path):
    return [json.loads(line) for line in pathlib.Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]

print(f"MOCK = {MOCK}  (1 = canned retriever/agent/judge/simulator)")
if not MOCK and not os.getenv("ANTHROPIC_API_KEY"):
    raise SystemExit("MOCK=0 needs ANTHROPIC_API_KEY in your environment (.env).")

## 1. RAG: split retrieval from generation

Retrieval is graded **classically** against labeled relevant chunks — recall@k, precision@k,
and **MRR** (how high the first relevant chunk ranks). Generation is graded by **faithfulness**
(is every claim supported by retrieved context?) and **answer-relevance** (does it address the
question?), both judgeable by rubric. The decomposition tells you whether to fix the
*retriever* or the *prompt* — the single most common fork in RAG debugging.

In [ ]:
corpus = {c["chunk_id"]: c["text"] for c in load_jsonl(DATA / "rag-corpus.jsonl")}
rag_cases = load_jsonl(DATA / "rag-cases.jsonl")
print(f"{len(corpus)} corpus chunks, {len(rag_cases)} rag cases")

def mock_retriever(question, k=3):
    """Canned, seeded retriever. Keyword overlap with a little injected noise so
    retrieval is imperfect (otherwise the eval has nothing to catch)."""
    q = set(question.lower().replace("?", "").split())
    scored = []
    for cid, text in corpus.items():
        overlap = len(q & set(text.lower().split()))
        scored.append((overlap + random.random() * 0.4, cid))   # seeded jitter
    scored.sort(reverse=True)
    return [cid for _, cid in scored[:k]]

def recall_at_k(retrieved, relevant):
    rel = set(relevant)
    return len(set(retrieved) & rel) / len(rel) if rel else 0.0

def precision_at_k(retrieved, relevant):
    rel = set(relevant)
    return len(set(retrieved) & rel) / len(retrieved) if retrieved else 0.0

def mrr(retrieved, relevant):
    rel = set(relevant)
    for i, cid in enumerate(retrieved, start=1):
        if cid in rel:
            return 1.0 / i
    return 0.0

for case in rag_cases[:3]:
    got = mock_retriever(case["question"])
    print(f"{case['id']}: recall={recall_at_k(got, case['relevant_chunks']):.2f} "
          f"prec={precision_at_k(got, case['relevant_chunks']):.2f} "
          f"mrr={mrr(got, case['relevant_chunks']):.2f}  retrieved={got}")

### 🔮 Predict

Take `rag-002` ("Can I get a refund just because I want one?"). Suppose the agent answers
*"Yes, just ask support."* — **wrong**. Before running: is that a **retrieval miss** (the
policy chunk never made it into context) or a **generation miss** (the chunk was retrieved but
the model contradicted it)? Read the decomposed scores below and see which fork to fix.

In [ ]:
def faithfulness_judge(answer, context_texts):
    """Rubric judge for groundedness. MOCK: canned -> claim must be supported by some
    retrieved chunk. Here we check the wrong 'yes a refund' answer against policy text."""
    if MOCK:
        ctx = " ".join(context_texts).lower()
        contradicts = "just ask" in answer.lower() and "only for" in ctx
        return not contradicts
    raise NotImplementedError("live judge omitted in companion; see 22-01 judge() shape")

case = next(c for c in rag_cases if c["id"] == "rag-002")
retrieved = mock_retriever(case["question"])
context_texts = [corpus[cid] for cid in retrieved if cid in corpus]

wrong_answer = "Yes, just ask support and they'll refund you."
retrieval_hit = recall_at_k(retrieved, case["relevant_chunks"]) > 0
faithful = faithfulness_judge(wrong_answer, context_texts)

print(f"retrieved chunks: {retrieved}")
print(f"retrieval found a relevant chunk? {retrieval_hit}")
print(f"generation faithful to context?   {faithful}")
print("Verdict:", "GENERATION miss (had the policy, ignored it)" if retrieval_hit and not faithful
      else "RETRIEVAL miss (never saw the policy)")

**What you just saw.** The retriever *did* surface the policy chunk, yet the answer
contradicted it — a **generation** failure. You'd fix the prompt (or add a faithfulness gate),
not the retriever. The same wrong final answer with `retrieval_hit=False` would have sent you
to chunking/embeddings instead. One score, two very different afternoons.

## 2. Trajectory evals: grade the path, not just the destination

An agent's **trajectory** is its sequence of steps and tool calls. Final-answer grading misses
the agent that succeeded *expensively* or *unsafely*. Score the path on four axes:
**selection** (right tools), **efficiency** (steps/tokens/cost vs a budget), **recovery**
(re-plan after a failed call rather than hallucinate a result), and **path-safety** (no
forbidden tools; asked for approval where policy demands).

In [ ]:
# A canned trajectory: list of steps, each a tool call (or a failure + recovery).
def mock_trajectory(case_id):
    """Seeded, deterministic agent traces keyed by trajectory case id."""
    traces = {
        "traj-001": [
            {"tool": "get_invoices", "ok": True},
            {"tool": "issue_refund", "ok": True},
        ],
        "traj-002": [
            {"tool": "search_docs", "ok": False},      # first call fails...
            {"tool": "search_docs", "ok": True},        # ...agent retries (recovery)
        ],
        "traj-003": [
            {"tool": "get_invoices", "ok": True},        # correctly stops, no refund
        ],
        "traj-004": [
            {"tool": "get_invoices", "ok": True},
            {"tool": "get_invoices", "ok": True},        # redundant repeat -> inefficiency
            {"tool": "get_invoices", "ok": True},
        ],
    }
    return traces[case_id]

def trajectory_score(case, trace):
    tools_used = [s["tool"] for s in trace]
    expected_tools = [c["name"] for c in case["expected_calls"]]
    failed = [i for i, s in enumerate(trace) if not s["ok"]]
    recovered = all(
        any(trace[j]["ok"] and trace[j]["tool"] == trace[i]["tool"] for j in range(i + 1, len(trace)))
        for i in failed
    )
    return {
        "selection_ok": set(expected_tools).issubset(tools_used),
        "within_budget": len(trace) <= case["budget_steps"],
        "recovered": recovered,                              # vacuously True if nothing failed
        "path_safe": not any(t in case["forbidden_tools"] for t in tools_used),
    }

traj_cases = load_jsonl(DATA / "trajectory-cases.jsonl")
for c in traj_cases:
    print(c["id"], trajectory_score(c, mock_trajectory(c["id"])))

Read the output: `traj-002` shows **recovery** (a failed `search_docs` retried and
succeeded), and `traj-004` trips **`within_budget=False`** — it answered correctly but called
`get_invoices` three times when one would do. Final-answer grading would have called both a
clean pass.

## 3. Tool-call accuracy: the arguments are the hard half

"Right tool, right order" is the easy half. The hard half is the **arguments**: a call can be
schema-valid yet wrong — `get_invoices(month="June")` when the user said May *parses fine and
fails the task*. This is the book's `tool_call_score`: right tool, valid args, **right
argument values**, no hallucinated tool/param, no over-calling.

In [ ]:
KNOWN_TOOLS = {"get_invoices", "issue_refund", "search_docs"}   # the registry
SCHEMAS = {
    "get_invoices": {"account", "month"},
    "issue_refund": {"account", "invoice", "amount"},
    "search_docs": {"query"},
}

def schema_ok(name, args):
    return name in SCHEMAS and set(args) == SCHEMAS[name]

def tool_call_score(case, expected, actual):
    """The book's shape. `expected`/`actual` are dicts with 'name' and 'args'."""
    expected_tools = {c["name"] for c in case["expected_calls"]}
    return {
        "right_tool": actual["name"] == expected["name"],
        "valid_args": schema_ok(actual["name"], actual["args"]),
        "right_values": actual["args"] == expected["args"],
        "no_hallucination": actual["name"] in KNOWN_TOOLS,
        "not_overcalling": actual["name"] in expected_tools,
    }

case = next(c for c in traj_cases if c["id"] == "traj-001")
expected = case["expected_calls"][0]                       # get_invoices, month=May
schema_valid_but_wrong = {"name": "get_invoices", "args": {"account": "A-1009", "month": "June"}}

print("expected call:", expected)
print("actual call  :", schema_valid_but_wrong)
print("score:", tool_call_score(case, expected, schema_valid_but_wrong))

### ⚠️ Pitfall — a schema-valid call can still be wrong

Above, `valid_args` is **True** (the JSON has the right keys) but `right_values` is **False**
(`month="June"` when the case expects May). If you only check the tool name and schema, this
bug ships. **Always score argument *values* against the case's expected call.**

## 4. User-simulator: evaluating a multi-turn agent

A conversation branches — turn three only exists in response to what the agent did on turn two,
so a frozen `(input, expected)` set can't exercise it. You need a **user-simulator**: an LLM
given a **persona**, a **goal**, and **constraints**, that opens the conversation, reacts to
what the agent *actually said*, and loops until the goal is met or a turn cap. Then you grade
the whole trajectory **and the final side effects** — was the refund actually issued, and only
within policy? This is the book's `simulate` loop, made local and canned.

In [ ]:
# Canned persona-simulator + canned agent so the multi-turn loop is deterministic.
class MockSimulator:
    """Persona: a customer with a genuine duplicate charge who won't share the account
    number until asked. Pinned to its persona so it never solves the task for the agent."""
    def __init__(self, goal):
        self.goal = goal
        self.script = [
            "I was charged twice for May, can you fix it?",     # opening
            "Sure, my account is A-1009.",                        # reacts: gives number when asked
        ]
        self.i = 0

    def opening(self):
        self.i = 1
        return self.script[0]

    def react(self, agent_msg):
        # Stays in character: only answers what a real user would, never over-helps.
        if "account" in agent_msg.lower() and self.i < len(self.script):
            msg = self.script[self.i]; self.i += 1
            return msg
        return "Okay, thanks."

    def goal_met(self, side_effects):
        return side_effects.get("refund_issued") and side_effects.get("within_policy")

class MockAgent:
    """Asks for the account, looks it up, issues the in-policy refund on the duplicate."""
    def __init__(self):
        self.side_effects = {"refund_issued": False, "within_policy": False}
    def respond(self, history, user_msg):
        if "A-1009" in user_msg:
            self.side_effects = {"refund_issued": True, "within_policy": True}
            return "Thanks — I see the duplicate May charge and have requested a $40 refund per policy."
        return "I can help. What is your account number so I can look up the invoices?"

def simulate(agent, sim, max_turns=8):
    """The book's simulate loop: persona opens, agent responds, persona reacts, repeat."""
    history, turns = [], 0
    user_msg = sim.opening()
    while turns < max_turns:
        agent_msg = agent.respond(history, user_msg)
        history += [("user", user_msg), ("agent", agent_msg)]
        if sim.goal_met(agent.side_effects):
            break
        user_msg = sim.react(agent_msg)
        turns += 1
    return {"transcript": history, "turns": turns,
            "side_effects": agent.side_effects,
            "resolved": sim.goal_met(agent.side_effects)}

result = simulate(MockAgent(), MockSimulator(goal="refund the duplicate May charge"))
for role, msg in result["transcript"]:
    print(f"{role:6s}: {msg}")
print(f"\nresolved={result['resolved']}  side_effects={result['side_effects']}  turns={result['turns']}")

### ⚠️ Pitfall — the simulator breaks character

A strong model will *helpfully* break character — answering questions the persona wouldn't
know or quietly solving the task for your agent — which **manufactures success** that won't
survive real users. Pin it with a strict system prompt, use a **different, cheaper** model than
the agent, and **validate simulated transcripts against real ones**. Grade the **final side
effects** (was the refund actually issued, within policy?), not just whether the chat *sounded*
resolved.

## 5. Harvesting evals from production (with PII redaction)

Your real input distribution lives in production. Map signals → eval data: thumbs-down,
rephrase/retry, **a human edit of a draft = a free gold label**, escalation, low-confidence.
Keep a **random baseline** sample *plus* **slice-stratified targeted mining**. And before
*anything* lands in a dataset, **redact PII** — raw traffic is radioactive until scrubbed.

In [ ]:
import re

PII_PATTERNS = [
    (re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+"), "[EMAIL]"),
    (re.compile(r"\b\d{12,19}\b"), "[CARD]"),
    (re.compile(r"\bA-\d{4,}\b"), "[ACCOUNT]"),
]

def redact(text):
    for pat, repl in PII_PATTERNS:
        text = pat.sub(repl, text)
    return text

SIGNAL_TO_USE = {
    "thumbs_down": "mine as a candidate failure case (never the headline metric)",
    "rephrase": "pair (failed answer, eventual success) into a hard case",
    "human_edit": "the edited draft IS the gold reference",
    "escalation": "high-value case; label with the human's resolution",
    "low_confidence": "sample for review before it becomes a visible failure",
}

raw_trace = {
    "signal": "human_edit",
    "input": "I was double charged, email me at jane.doe@example.com about account A-1009",
    "human_edited_answer": "Confirmed the duplicate on account A-1009; a $40 refund was requested.",
}

def trace_to_case(trace):
    """Convert a production trace into a (redacted) golden case, tagged with its pool."""
    return {
        "input": redact(trace["input"]),
        "expected_behavior": redact(trace["human_edited_answer"]),
        "source_signal": trace["signal"],
        "pool": "eval",                      # disjoint from few-shot/train (see 22-01)
        "tags": ["billing", "harvested"],
    }

print("signal -> use:")
for k, v in SIGNAL_TO_USE.items():
    print(f"  {k:14s} {v}")
print("\nharvested case (PII redacted):")
print(json.dumps(trace_to_case(raw_trace), indent=2))

### ⚠️ Pitfall — raw production traffic is radioactive

Notice the email and account number are gone before the case is stored. **Anonymize/redact
before anything lands in a dataset** — names, emails, account/card numbers — and honor the
consent/retention rules (Ch 41). A golden set is replayed for years; never let raw PII into it.

## 6. Benchmarks: read them for the oracle, not the score

A senior reads public benchmarks fluently — to interpret release notes and **borrow harness
designs**, not to chase a leaderboard. Read each as *"what oracle does it use?"*: τ-bench
(user-simulator + final DB state — you just built a local version), SWE-bench (execution: run
the repo's tests), WebArena/GAIA (multi-step web), BFCL (function-call accuracy via AST — the
formalization of `tool_call_score`), AgentBench (breadth). **Borrow the oracle, not the
score** — and never forget **contamination** and **distribution mismatch**: no public set
matches *your* users, tools, or policies.

## 🎯 Senior lens

Triage by **distribution, not drama**. The dramatic one-off failure that everyone Slacks about
is rarely the biggest lever; the boring cluster of fifteen near-identical retrieval misses is.
Read the *slice* table, fix the biggest failure cluster first, and file the spectacular one-off
as a single eval case so it can never silently return. Drama is a poor prioritizer; the
distribution is an honest one.

## Recap

- **Decompose**: RAG into retrieval (recall/precision@k, MRR) vs generation (faithfulness,
  relevance) — opposite fixes.
- **Trajectory** evals grade the *path*: selection, efficiency vs budget, recovery, path-safety.
- **Tool-call accuracy** must check argument **values**, not just the tool name — a
  schema-valid call can be wrong.
- A **user-simulator** (persona + goal + constraints) tests multi-turn agents; grade the
  **final side effects** and keep the simulator in character.
- **Harvest** production into eval cases by signal, sample random + slice-stratified, and
  **redact PII** before storage.

## Exercises

**1.** Add a `rag-cases.jsonl` entry whose relevant chunk the `mock_retriever` *misses* (use
words absent from the chunk). Predict whether the failure now reads as retrieval or generation,
then run the decomposition.

**2.** Add a trajectory case where the agent calls a **forbidden** tool. Predict which axis of
`trajectory_score` flips, then confirm.

**3.** Give `MockSimulator` a constraint that it **gives up after two stonewalls**. Predict
whether `resolved` becomes `False` if `MockAgent` never asks for the account, then run it.

**4.** Add a phone-number pattern to `PII_PATTERNS`. Predict the redacted output for an input
containing a phone number, then confirm `trace_to_case` scrubs it.

## Next

- **Next notebook:** [`22-03-eval-harness-and-ci-gate.ipynb`](22-03-eval-harness-and-ci-gate.ipynb)
  — wire these scorers into the capstone's `evals/` package with a concurrent harness and a
  **CI gate** that blocks regressions.
- **Blueprint (production version):** [`../../../blueprints/eval-harness/`](../../../blueprints/eval-harness/).
- **Template:** [`../../../templates/eval-dataset-template/`](../../../templates/eval-dataset-template/)
  — the tagged case schema, now including the eval-holdout vs few-shot/train partition.
- **Capstone:** the trajectory + tool-call scorers feed `capstone/evals/`.